<a href="https://colab.research.google.com/github/rahna1369/Supply-Chain-AI-Dashboard/blob/main/supply_chain_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.6 MB/s eta 0:00:00


## Import Libraries

In [2]:
# Import necessary libraries for Deep Learning, Data Processing, and Hyperparameter Tuning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, classification_report

# Suppress Optuna verbose logs for a cleaner training output
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Libraries imported successfully!")

Libraries imported successfully!


**Explanation:** This cell imports all core libraries: PyTorch for building and training neural networks, Pandas/NumPy for data manipulation, Scikit-Learn for data preprocessing pipelines, and Optuna for automated hyperparameter tuning.

## Load and Preprocess the CSV Dataset

In [3]:
# Load the CSV dataset
file_path = 'global_supply_chain_risk_2026.csv'

try:
    df = pd.read_csv(file_path).drop_duplicates()
    print("CSV loaded successfully!")
    print("\n--- Actual Columns Found in Your CSV ---")
    print(df.columns.tolist())
    print("-" * 40)
except FileNotFoundError:
    print(f"File not found at {file_path}")



CSV loaded successfully!

--- Actual Columns Found in Your CSV ---
['Shipment_ID', 'Date', 'Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index', 'Geopolitical_Risk_Score', 'Weather_Condition', 'Carrier_Reliability_Score', 'Lead_Time_Days', 'Disruption_Occurred']
----------------------------------------


In [4]:
df

,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days,Disruption_Occurred
0,SC-10000,16-10-2025,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,SC-10001,24-04-2024,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,SC-10002,26-01-2024,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,SC-10003,08-10-2024,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,SC-10004,07-09-2024,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,SC-14995,10-11-2025,Rotterdam,Dubai,Rail,Perishables,13512.26,428.97,3.35,8.8,Fog,0.664,20.28,0
4996,SC-14996,23-11-2025,Dubai,Shanghai,Air,Perishables,2231.02,448.86,1.79,0.5,Clear,0.863,0.50,0
4997,SC-14997,18-10-2025,Antwerp,Busan,Rail,Pharmaceuticals,5253.72,473.41,1.37,3.0,Clear,0.962,4.41,0
4998,SC-14998,29-01-2025,Singapore,Dubai,Rail,Automotive,12328.31,199.35,3.45,4.4,Storm,0.819,31.18,1


In [5]:
# Define shared features and target variables
features = [
    'Geopolitical_Risk',
    'Carrier_Reliability_Score',
    'Weather_Condition',
    'Transport_Mode',
    'Product_Category',
    'Distance_km',
    'Weight_MT',
    'Fuel_Price_Index'
]

target_regression = 'Lead_Time_Days'
target_classification = 'Disruption'

In [6]:
# Safety check for missing columns
required_cols = features + [target_regression, target_classification]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"\n[ERROR] These columns are missing from your CSV: {missing_cols}")
    print("Please update the names in the 'features' list above to match your CSV headers exactly.")
else:
    X = df[features]
    y_reg = df[target_regression].values.astype(np.float32)
    y_cls = df[target_classification].values.astype(np.float32)


[ERROR] These columns are missing from your CSV: ['Geopolitical_Risk', 'Disruption']
Please update the names in the 'features' list above to match your CSV headers exactly.


In [7]:
# Automatically filter out any features missing from the dataframe
numeric_features = ['Geopolitical_Risk', 'Carrier_Reliability_Score', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index']
categorical_features = ['Weather_Condition', 'Transport_Mode', 'Product_Category']

valid_numeric = [col for col in numeric_features if col in df.columns]
valid_categorical = [col for col in categorical_features if col in df.columns]

print(f"Valid Numeric Features: {valid_numeric}")
print(f"Valid Categorical Features: {valid_categorical}")

# Recreate preprocessor using only valid features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, valid_numeric),
        ('cat', categorical_transformer, valid_categorical)
    ]
)

# Create X with only available columns
valid_features = valid_numeric + valid_categorical
X = df[valid_features]

# Transform features into dense numpy array
X_processed = preprocessor.fit_transform(X)
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

input_dim = X_processed.shape[1]
print(f"\nData preprocessing complete! Total feature dimensions: {input_dim}")

Valid Numeric Features: ['Carrier_Reliability_Score', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index']
Valid Categorical Features: ['Weather_Condition', 'Transport_Mode', 'Product_Category']

Data preprocessing complete! Total feature dimensions: 18


**Explanation:** This cell reads your CSV file, drops duplicate rows, separates features from targets (Lead_Time_Days and Disruption), fills missing values, normalizes numerical numbers using standard scaling, and converts text categories into binary inputs via one-hot encoding.

In [8]:
# Check exact numeric column ranges in your original dataset
numeric_cols = ['Carrier_Reliability_Score', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index']
valid_cols = [col for col in numeric_cols if col in df.columns]

print("True Dataset Summary:")
display(df[valid_cols].describe().T[['mean', 'min', '50%', 'max']])

True Dataset Summary:


,mean,min,50%,max
Carrier_Reliability_Score,0.754387,0.50,0.757,1.00
Distance_km,7704.063888,500.17,7750.125,14995.91
Weight_MT,246.252052,1.03,243.500,499.75
Fuel_Price_Index,2.854552,1.20,2.840,4.50


## Train-Test Split

In [9]:
# Print your columns to find your exact target names
print("Available columns in your CSV:", df.columns.tolist())

# REPLACE these strings with the exact target column names shown in your printed list above:
target_regression = df.columns[-2]      # Update with your actual regression column name
target_classification = df.columns[-1]  # Update with your actual classification column name

y_reg = df[target_regression].values.astype(np.float32)
y_cls = df[target_classification].values.astype(np.float32)

# Split data into training and testing sets (80% train, 20% test)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_processed, y_reg, test_size=0.2, random_state=42
)

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_processed, y_cls, test_size=0.2, random_state=42
)

print("Data successfully split into training and testing sets.")

Available columns in your CSV: ['Shipment_ID', 'Date', 'Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index', 'Geopolitical_Risk_Score', 'Weather_Condition', 'Carrier_Reliability_Score', 'Lead_Time_Days', 'Disruption_Occurred']
Data successfully split into training and testing sets.


**Explanation:** We partition the dataset into an 80% training set (for the neural network to learn patterns) and a 20% testing set (to evaluate performance on unseen shipments).

## Optuna Tuning & Training for Regression Model (Lead Time Days)

In [10]:
# Define dynamic neural network architecture for regression
class TunableRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate):
        super(TunableRegressor, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.layer3 = nn.Linear(hidden_dim // 2, 1) # Linear output for continuous values

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.layer3(x)
        return x

In [11]:
# Optuna objective function to find best hyperparameters for Regression
def objective_reg(trial):
    hidden_dim = trial.suggest_int('hidden_dim', 32, 128, step=32)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4, step=0.1)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train_reg, dtype=torch.float32),
            torch.tensor(y_train_reg, dtype=torch.float32).unsqueeze(1)
            ),
        batch_size=batch_size, shuffle=True)
    test_tensor = torch.tensor(X_test_reg, dtype=torch.float32)

    model = TunableRegressor(input_dim, hidden_dim, dropout_rate)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(15):
        for bx, by in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(test_tensor).numpy().flatten()
    rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
    return rmse # Minimize Root Mean Squared Error

print("Starting Optuna Hyperparameter Tuning for Regression...")
study_reg = optuna.create_study(direction='minimize')
study_reg.optimize(objective_reg, n_trials=10)
print(f"Best Regression RMSE : {study_reg.best_value:.2f} days")
print("Best Regressor Hyperparameters:", study_reg.best_params)

Starting Optuna Hyperparameter Tuning for Regression...
Best Regression RMSE : 4.59 days
Best Regressor Hyperparameters: {'hidden_dim': 96, 'dropout_rate': 0.4, 'lr': 0.0015328201029954614, 'batch_size': 32}


In [12]:
# Train final regression model using the best discovered parameters
best_params_reg = study_reg.best_params
final_reg_model = TunableRegressor(input_dim, best_params_reg['hidden_dim'], best_params_reg['dropout_rate'])
optimizer_reg = optim.Adam(final_reg_model.parameters(), lr=best_params_reg['lr'])
criterion_reg = nn.MSELoss()

train_loader_reg = DataLoader(
    TensorDataset(
        torch.tensor(X_train_reg, dtype=torch.float32),
        torch.tensor(y_train_reg, dtype=torch.float32 ).unsqueeze(1)
        ),
    batch_size=best_params_reg['batch_size'],
    shuffle=True)

final_reg_model.train()
for epoch in range(25):
    for bx, by in train_loader_reg:
        optimizer_reg.zero_grad()
        loss = criterion_reg(final_reg_model(bx), by)
        loss.backward()
        optimizer_reg.step()

final_reg_model.eval()
with torch.no_grad():
    y_pred_reg = final_reg_model(torch.tensor(X_test_reg, dtype=torch.float32)).numpy().flatten()

print(f"Final Regression MAE : {mean_absolute_error(y_test_reg, y_pred_reg):.2f} days")

Final Regression MAE : 2.30 days


**Explanation:** This cells builds a neural network regressor to predict continuous delivery lead times. It uses Optuna to search for optimal learning rates, hidden dimensions, and batch sizes, minimizing prediction error (RMSE), and then trains the final model.

 ## Optuna Tuning & Training for Classification Model (Disruption Status)

In [13]:
# Define dynamic neural network architecture for classification
class TunableClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate):
        super(TunableClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.layer3 = nn.Linear(hidden_dim // 2, 1)
        self.sigmoid = nn.Sigmoid() # Sigmoid for binary probability output

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

In [14]:
# Optuna objective function to find best hyperparameters for Classification
def objective_cls(trial):
    hidden_dim = trial.suggest_int('hidden_dim', 32, 128, step=32)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4, step=0.1)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train_cls, dtype=torch.float32),
            torch.tensor(y_train_cls, dtype=torch.float32).unsqueeze(1)),
         batch_size=batch_size,
         shuffle=True)
    test_tensor = torch.tensor(X_test_cls, dtype=torch.float32)

    model = TunableClassifier(input_dim, hidden_dim, dropout_rate)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(15):
        for bx, by in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = (model(test_tensor) >= 0.5).float().numpy().flatten()
    acc = accuracy_score(y_test_cls, preds)
    return acc # Maximize Accuracy

print("\nStarting Optuna Hyperparameter Tuning for Classification...")
study_cls = optuna.create_study(direction='maximize')
study_cls.optimize(objective_cls, n_trials=10)

print(f"Best Classification Accuracy : {study_cls.best_value * 100:.2f}%")
print("Best Classifier Hyperparameters:", study_cls.best_params)



Starting Optuna Hyperparameter Tuning for Classification...
Best Classification Accuracy : 71.60%
Best Classifier Hyperparameters: {'hidden_dim': 128, 'dropout_rate': 0.2, 'lr': 0.00038191489375146993, 'batch_size': 64}


In [15]:
# Train final classification model using the best discovered parameters
best_params_cls = study_cls.best_params
final_cls_model = TunableClassifier(input_dim, best_params_cls['hidden_dim'], best_params_cls['dropout_rate'])
optimizer_cls = optim.Adam(final_cls_model.parameters(), lr=best_params_cls['lr'])
criterion_cls = nn.BCELoss()

train_loader_cls = DataLoader(
    TensorDataset(
        torch.tensor(X_train_cls, dtype=torch.float32),
        torch.tensor(y_train_cls, dtype=torch.float32).unsqueeze(1)),
    batch_size=best_params_cls['batch_size'],
    shuffle=True)

final_cls_model.train()
for epoch in range(25):
    for bx, by in train_loader_cls:
        optimizer_cls.zero_grad()
        loss = criterion_cls(final_cls_model(bx), by)
        loss.backward()
        optimizer_cls.step()

final_cls_model.eval()
with torch.no_grad():
    outputs = final_cls_model(torch.tensor(X_test_cls, dtype=torch.float32))
    y_pred_cls = (outputs >= 0.5).float().numpy().flatten()

print(f"\nFinal Classification Accuracy : {accuracy_score(y_test_cls, y_pred_cls) * 100:.2f}%\n")
print("Classification Report:\n", classification_report(y_test_cls, y_pred_cls))


Final Classification Accuracy : 71.20%

Classification Report:
               precision    recall  f1-score   support

         0.0       0.58      0.83      0.69       379
         1.0       0.86      0.64      0.73       621

    accuracy                           0.71      1000
   macro avg       0.72      0.73      0.71      1000
weighted avg       0.76      0.71      0.72      1000



**Explanation:** This cells builds a neural network classifier using a Sigmoid output layer and Binary Cross-Entropy Loss to predict whether a shipment faces disruption. Optuna automatically tunes its architecture to maximize overall test accuracy.

In [16]:
def to_1d(arr):
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    return np.array(arr).reshape(-1)

# Convert all variables to 1D arrays
y_reg_arr = to_1d(y_reg)
y_pred_reg_arr = to_1d(y_pred_reg)
y_cls_arr = to_1d(y_cls)
y_pred_cls_arr = to_1d(y_pred_cls)

# Print array lengths to identify mismatch
print(f"Actual Lead Time length: {len(y_reg_arr)}")
print(f"Predicted Lead Time length: {len(y_pred_reg_arr)}")
print(f"Actual Disruption length: {len(y_cls_arr)}")
print(f"Predicted Disruption length: {len(y_pred_cls_arr)}")

# Find common test length and slice arrays to match
min_len = min(len(y_reg_arr), len(y_pred_reg_arr), len(y_cls_arr), len(y_pred_cls_arr))

output_df = pd.DataFrame({
    'Actual_Lead_Time': y_reg_arr[:min_len],
    'Predicted_Lead_Time': y_pred_reg_arr[:min_len],
    'Actual_Disruption': y_cls_arr[:min_len],
    'Predicted_Disruption': y_pred_cls_arr[:min_len]
})

# Save to CSV
csv_filename = 'tuned_deep_learning_supply_chain_output.csv'
output_df.to_csv(csv_filename, index=False)
print(f"\n✅ Successfully saved {csv_filename} with {min_len} rows!")

Actual Lead Time length: 5000
Predicted Lead Time length: 1000
Actual Disruption length: 5000
Predicted Disruption length: 1000

✅ Successfully saved tuned_deep_learning_supply_chain_output.csv with 1000 rows!


In [17]:
import os
import joblib

# 1. Set PyTorch models to evaluation mode
final_reg_model.eval()
final_cls_model.eval()

# Check if output_df exists before packaging
df_to_save = output_df if 'output_df' in globals() or 'output_df' in locals() else None

# 2. Package all trained models, preprocessor, and output dataframe
pipeline_data = {
    'reg_model': final_reg_model,
    'cls_model': final_cls_model,
    'preprocessor': preprocessor,
    'valid_numeric': valid_numeric if 'valid_numeric' in globals() else [],
    'valid_categorical': valid_categorical if 'valid_categorical' in globals() else [],
    'output_df': df_to_save
}

# 3. Export everything into ONE joblib file
file_path = 'supply_chain_pipeline.joblib'
joblib.dump(pipeline_data, file_path)

# 4. Save CSV file for visualization
if df_to_save is not None:
    df_to_save.to_csv('tuned_deep_learning_supply_chain_output.csv', index=False)
    print("CSV exported successfully!")
else:
    print("Warning: 'output_df' was not found! Make sure you ran the cell that generates predictions first.")

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f" Saved '{file_path}' ({size_mb:.2f} MB) and CSV successfully!")

CSV exported successfully!
 Saved 'supply_chain_pipeline.joblib' (0.09 MB) and CSV successfully!


In [18]:
import joblib
import pandas as pd
import torch

pipeline = joblib.load('supply_chain_pipeline.joblib')
reg_model = pipeline['reg_model']
cls_model = pipeline['cls_model']
preprocessor = pipeline['preprocessor']

reg_model.eval()
cls_model.eval()

# 🎯 Updated with realistic values based on your dataset summary table:
sample_shipment = {
    'Carrier_Reliability_Score': [0.75],    # Dataset median ~ 0.75
    'Distance_km': [7750.0],                # Dataset median ~ 7,750 km
    'Weight_MT': [243.5],                   # Dataset median ~ 243 MT
    'Fuel_Price_Index': [2.84],             # Dataset median ~ 2.84
    'Weather_Condition': ['Clear'],
    'Transport_Mode': ['Road'],
    'Product_Category': ['Electronics']
}

input_df = pd.DataFrame(sample_shipment)
processed_features = preprocessor.transform(input_df)
if hasattr(processed_features, "toarray"):
    processed_features = processed_features.toarray()

tensor_input = torch.tensor(processed_features, dtype=torch.float32)

with torch.no_grad():
    pred_lead_time = reg_model(tensor_input).item()
    disruption_prob = cls_model(tensor_input).item()

print(f"🎯 Accurate Prediction -> Lead Time: {pred_lead_time:.0f} Days | Disruption Risk: {disruption_prob * 100:.2f}%")

🎯 Accurate Prediction -> Lead Time: 6 Days | Disruption Risk: 38.51%
